In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import glob
import pickle
from tqdm import tqdm
import powerlaw
import matplotlib as mpl
from datetime import datetime, timedelta
from tqdm import tqdm
import textacy
from functools import partial
from fuzzywuzzy import fuzz
import re

data_directory = os.getcwd()[:-4] + 'original_data/'


# text_font = 18
# latex_preamble = r"\usepackage{times} \usepackage{amsmath} \usepackage{amssymb}"
# mpl.rcParams.update({
#                     'text.usetex': True,
#                     'font.size': 18, 
#                     'font.style': 'normal',
#                     'font.family':'serif',
#                     'text.latex.preamble': latex_preamble,
#                     'font.serif': ['Times']})

In [2]:
# !python -m spacy download en_core_web_sm
# !pip install textacy

In [23]:
'gabri'.replace('b*','', regex=True)

TypeError: str.replace() takes no keyword arguments

In [26]:
df = pd.read_csv(data_directory + 'VideoInfo_ABC.csv')

# df['Title'] = df['Title'].apply(lambda x: re.sub(r'\|.*', '', x))

df['Title'] = df['Title'].str.replace(r'\|.*', '', regex=True)

# df['Tokens'] = df['Title'].apply(get_tokens)

In [4]:
def get_tokens(video_title):
    # Create the corpus
    corpus = textacy.Corpus('en_core_web_sm', data=video_title)

    # Extract terms and entities
    docs_terms = (
        textacy.extract.terms(
            doc,
            # Extract n-grams of varying lengths with specified POS
            ngs=partial(textacy.extract.ngrams, n=(1,2,3,4,5,6), include_pos={"NOUN", "ADJ", "ADV"}),  
            # Extract specified entity types
            ents=partial(textacy.extract.entities, include_types={"ORG", "GPE", "LOC", "PERSON", "NORP", "FAC", "PRODUCT", "EVENT", "WORK_OF_ART", "LAW", "LANGUAGE", "DATE", "TIME", "PERCENT", "MONEY", "QUANTITY", "ORDINAL", "CARDINAL"}))
        for doc in corpus  # Iterate over each document in the corpus
    )
    
    # Convert terms to strings
    tokenized_docs = (
        list(textacy.extract.terms_to_strings(doc_terms, by="lemma"))
        for doc_terms in docs_terms
    )

    # Convert the generator to a list
    tokenized_words_list = list(tokenized_docs)

    # Filter out n-grams that are substrings of longer n-grams
    filtered_ngrams = []
    for doc in tokenized_words_list:
        longest_ngrams = set(doc)
        for ngram in doc:
            for other_ngram in doc:
                if ngram != other_ngram and ngram in other_ngram:
                    longest_ngrams.discard(ngram)
                    break
        filtered_ngrams.append(list(longest_ngrams))

    return filtered_ngrams[0]

In [5]:
# df.iloc[1]['Title']

'CDC releases new guidance for those who are fully vaccinated'

In [6]:
get_tokens('white house press briefing with press secretary jen psaki')

[['white house', 'press briefing', 'jen psaki']]

In [11]:
df['Tokens'] = df['Tokens'].apply(lambda x: x[0])

In [14]:
final_list = sum(df['Tokens'], [])

final_list

['March 9',
 'COVID-19: World in Photos,',
 'Myanmar',
 'Justice',
 'Silent',
 'fully',
 'CDC',
 'new guidance',
 'felony trial attorney',
 'George Floyd',
 'murder trial',
 'jury selection',
 'trial',
 'delay',
 'Derek Chauvin',
 'Wednesday',
 'pandemic relief bill',
 'morning',
 '$1.9 trillion',
 'Biden',
 'Prince Harry',
 'statement',
 'Meghan interview l',
 'ABC News',
 'Buckingham Palace',
 'fully vaccinated people l',
 'new guideline',
 'CDC',
 'alleged hazing accident l',
 'college sophomore',
 'british monarchy',
 'Meghan Markle',
 'shocking interview',
 'football teammate',
 'teen boy',
 'sleepover',
 'black man',
 'George Floyd',
 'death',
 'practice',
 'headline today',
 'March 8, 2021',
 'family',
 'bombshell interview',
 'Harry',
 'Meghan',
 'credible',
 'Richard Besser',
 'new vaccine guideline',
 'guideline',
 'path',
 'CDC',
 'new',
 'million',
 'normal interaction',
 'trial',
 'spring break',
 'interview',
 'Harry',
 'Oprah',
 'ABC News',
 'Meghan',
 'Chauvin',
 'COVID

In [18]:
a, count = np.unique(final_list, return_counts=True)


#sort a by count
a = a[np.argsort(count)[::-1]]

count = count[np.argsort(count)[::-1]]

numbers = [x for x in a if re.search(r'\d', x) is not None]

numbers



['COVID-19',
 '2020',
 '2',
 '1',
 '20/20',
 '3',
 '4',
 '5',
 '2018',
 '8',
 '911',
 '10',
 '10%',
 '1st',
 '7',
 '6',
 '2019',
 'covid-19',
 '2017',
 '9/11',
 '9',
 '1st time',
 '20',
 '737',
 '6-year-old',
 '7-year-old',
 '12',
 '2016',
 '10-year-old',
 'covid-19 death',
 '9-year-old',
 '13-year-old',
 '30',
 '3-year-old',
 '2nd',
 '5-year-old',
 '8-year-old',
 '11-year-old',
 '12-year-old',
 '50',
 '20/20 l PART',
 '13',
 'covid-19 case',
 'Year 2019',
 '11',
 '15',
 '14-year-old',
 '50th anniversary',
 '50 year later',
 '2nd impeachment trial',
 'at least 2',
 '4-year-old',
 '17-year-old',
 '10 year',
 '2 week',
 '30 year',
 '2021',
 '100',
 '100,000',
 '2-year-old',
 'at least 1',
 '14',
 'More than 200',
 '29',
 '1969',
 '15-year-old',
 '16',
 '$1.9 trillion',
 '2 day',
 'at least 5',
 '75th anniversary',
 '1 year',
 '21',
 'Year 2018',
 '25 year',
 '23',
 '25',
 '60',
 '20 year',
 'Start of 2018',
 '200,000',
 '21-year-old',
 'Year 2020',
 '97',
 '16-year-old',
 'at least 3',
 

In [20]:
def calculate_fuzzy_match(tokens1, tokens2):
    match_scores = []
    for token1 in tokens1:
        for token2 in tokens2:
            match_score = fuzz.token_set_ratio(token1, token2)
            match_scores.append(match_score)
    return max(match_scores)

In [45]:
#create a list of all the unique tokens
unique_tokens = []
for tokens in df['Tokens']:
    for token in tokens:
        unique_tokens.append(token)

unique_tokens = list(set(unique_tokens))
unique_tokens = [x for x in unique_tokens if x != '']

unique_tokens = unique_tokens[:10]

#calculate the similarity between all the tokens
token_similarity = np.zeros((len(unique_tokens), len(unique_tokens)))
for i, token1 in tqdm(enumerate(unique_tokens)):
    for j, token2 in enumerate(unique_tokens):
        token_similarity[i, j] = fuzz.token_set_ratio(token1, token2)

token_similarity

#create sets of similar tokens if the number is higher than 90
similar_tokens = []
for i in range(token_similarity.shape[0]):
    good_ind = np.where(token_similarity[i] > 40)[0]
    if len(good_ind) > 1:
        similar_tokens.append([unique_tokens[j] for j in good_ind])#.append(unique_tokens[i]))

similar_tokens = [set(x) for x in similar_tokens]

#calculate intersections between the sets and if it is greater than 1, merge the sets
i = 0
while i < len(similar_tokens):
    j = i + 1
    while j < len(similar_tokens):
        if len(similar_tokens[i].intersection(similar_tokens[j])) > 0:
            similar_tokens[i] = similar_tokens[i].union(similar_tokens[j])
            similar_tokens.pop(j)
        else:
            j += 1
    i += 1

#create a dictionary with key being first value in the set and value being the rest of the set
similar_tokens_dict = {}
for tokens in similar_tokens:
    similar_tokens_dict[list(tokens)[0]] = list(tokens)[1:]

similar_tokens_dict

# use this dicitonary to replace the tokens in the dataframe
df['Tokens'] = df['Tokens'].apply(lambda x: [similar_tokens_dict[token] if token in similar_tokens_dict else token for token in x])

df['Tokens']



10it [00:00, 3855.06it/s]


0        [Myanmar, Silent, Justice, COVID-19: World in ...
1                         [release new, new guidance, CDC]
2        [felony trial, trial attorney, murder trial, f...
3             [jury selection, trial begin, Derek Chauvin]
4        [pandemic relief, relief bill, bill expect, Bi...
                               ...                        
19995     [wound aide, aide speak, baseball shooting, GOP]
19996           [baseball shooting, Gabby Giffords ', GOP]
19997    [congressman describe, baseball shooting, shoo...
19998    [critical condition, baseball practice, practi...
19999                 [survival emerge, rise fire, London]
Name: Tokens, Length: 20000, dtype: object

In [17]:


# Define a function to calculate fuzzy match score between two lists of tokens


# Perform fuzzy matching between tokens of different videos
fuzzy_match_scores = []
for i in range(len(df)):
    for j in range(i+1, len(df)):
        tokens1 = df.iloc[i]['Tokens']
        tokens2 = df.iloc[j]['Tokens']
        match_score = calculate_fuzzy_match(tokens1, tokens2)
        fuzzy_match_scores.append((i, j, match_score))

# Sort the fuzzy match scores in descending order
fuzzy_match_scores.sort(key=lambda x: x[2], reverse=True)

# Print the top matching pairs
for i, j, match_score in fuzzy_match_scores[:5]:
    print(f"Match Score: {match_score}")
    print(f"Video 1: {df.iloc[i]['Title']}")
    print(f"Video 2: {df.iloc[j]['Title']}")
    print()

ValueError: max() iterable argument is empty